In [ ]:
%pip install azure-ai-ml azure-identity scikit-learn joblib pandas -q

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

subscription_id = "<SUBSCRIPTION_ID>"
resource_group  = "<RESOURCE_GROUP>"
workspace_name  = "<WORKSPACE_NAME>"

ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id,
    resource_group,
    workspace_name,
)

print("Connected to workspace:", ml_client.workspace_name)

In [ ]:
import pandas as pd

df = pd.read_csv("telco_customer_churn.csv")

print("Shape:", df.shape)
print(df.dtypes)
df.head()

In [ ]:
print("Missing customerIDs:", df["customerID"].isna().sum())
print("Blank TotalCharges:", (df["TotalCharges"].astype(str).str.strip() == "").sum())
print("Churn distribution:\n", df["Churn"].value_counts(normalize=True))

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna(subset=["TotalCharges"]).reset_index(drop=True)

df = df.drop(columns=["customerID"])

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

print("Shape after cleaning:", df.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib

X = df.drop("Churn", axis=1)
y = df["Churn"]

categorical_cols = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod",
]
numeric_cols = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ("num", StandardScaler(), numeric_cols),
])

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline.fit(X_train, y_train)

preds = pipeline.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds, target_names=["stayed", "churned"]))

joblib.dump(pipeline, "model.pkl")
print("Pipeline (preprocessing + model) saved as model.pkl")

In [ ]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

model_entity = Model(
    path="model.pkl",
    type=AssetTypes.CUSTOM_MODEL,
    name="telco-churn-pipeline-model",
    description="Logistic Regression churn classifier (with preprocessing) on IBM Telco Customer Churn data",
)

registered_model = ml_client.models.create_or_update(model_entity)
print(f"Registered: {registered_model.name}, version {registered_model.version}")

In [ ]:
%%writefile score.py
import json
import os
import joblib
import pandas as pd

def init():
    global model
    model_path = os.path.join(os.getenv("AZUREML_MODEL_DIR"), "model.pkl")
    model = joblib.load(model_path)

def run(raw_data):
    payload = json.loads(raw_data)
    df = pd.DataFrame(payload["data"])
    predictions = model.predict(df)
    return predictions.tolist()

In [ ]:
%%writefile conda.yaml
name: telco-churn-env
channels:
  - defaults
dependencies:
  - python=3.10
  - pip
  - pip:
      - scikit-learn
      - joblib
      - azureml-defaults
      - pandas

In [ ]:
from azure.ai.ml.entities import Environment

env = Environment(
    name="telco-churn-env",
    conda_file="conda.yaml",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
)

In [ ]:
from azure.ai.ml.entities import ManagedOnlineEndpoint
import uuid, time

endpoint_name = "churn-endpoint-" + str(uuid.uuid4())[:8]

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Endpoint serving the Telco churn classifier",
    auth_mode="key",
)

ml_client.online_endpoints.begin_create_or_update(endpoint).result()

while True:
    state = ml_client.online_endpoints.get(endpoint_name).provisioning_state
    print("Endpoint provisioning state:", state)
    if state == "Succeeded":
        break
    if state == "Failed":
        raise RuntimeError("Endpoint provisioning failed.")
    time.sleep(10)

print("Endpoint ready:", endpoint_name)

In [ ]:
from azure.ai.ml.entities import ManagedOnlineDeployment, CodeConfiguration

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=endpoint_name,
    model=registered_model,
    environment=env,
    code_configuration=CodeConfiguration(
        code=".",
        scoring_script="score.py",
    ),
    instance_type="Standard_DS3_v2",
    instance_count=1,
)

ml_client.online_deployments.begin_create_or_update(deployment).result()

endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

print("Deployment complete.")

In [ ]:
import json

sample = X_test.iloc[[0]].to_dict(orient="records")
true_label = int(y_test.iloc[0])

with open("sample-request.json", "w") as f:
    json.dump({"data": sample}, f)

response = ml_client.online_endpoints.invoke(
    endpoint_name=endpoint_name,
    request_file="sample-request.json",
)
print("Prediction:", response)
print("Actual label:", true_label)

In [ ]:
endpoint_details = ml_client.online_endpoints.get(endpoint_name)
keys = ml_client.online_endpoints.get_keys(endpoint_name)

print("Scoring URI:", endpoint_details.scoring_uri)
print("Primary key:", keys.primary_key)

In [ ]:
ml_client.online_endpoints.begin_delete(name=endpoint_name).result()
print("Endpoint deleted.")